In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import requests
import zipfile
from pathlib import Path
import os

from src.data_helpers import ImageFolderCustom

In [2]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cpu


In [3]:
# Get data - subset of Food 101 dataset
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"
if image_path.is_dir():
  print(f"{image_path} dir already exists... skipping")
else:
  print(f"{image_path} dir doesn't exists... creating")
  image_path.mkdir(parents=True, exist_ok=True)

with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
  request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
  print("Downloading data...")
  f.write(request.content)

with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
  print("Unzipping...")
  zip_ref.extractall(image_path)

data\pizza_steak_sushi dir already exists... skipping
Unzipping...


In [4]:
# Setup train and test paths
train_dir = image_path / "train"
test_dir = image_path / "test"

# Transform all images to tensors

## Option 1: Use builtin dataset class

Here we use ImageFolder to create datasets

We can use it since our data dir is in standard image classification format

## Option 2: Creating a custom dataset class
This custom dataset will be a subclass of torch.utils.data.Dataset

We write our own custom dataloading function that has to be able to:
1. load images from file
2. get class names as list and as dict

We require standard data format.

Raise error if class names are not found.

-------------------------------------------------------

Pros:
- create dataset out of almost anything
- not limited to pre-built datasets

Cons
- more code = more performance issues and errors
- it doesn't mean it will work

In [5]:
# We can create separate transformations for train and test set
# Test set doesn't require data augmentation
train_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

In [6]:
# Test out ImageFolderCustom
train_data_custom = ImageFolderCustom(
    target_dir=train_dir,
    transform=train_transform
)
test_data_custom = ImageFolderCustom(
    target_dir=test_dir,
    transform=test_transform
)

print(train_data_custom, test_data_custom)

print()

# Check functionalities:
print(f"Get classes:\n {train_data_custom.classes}")
print(f"Get class-label dict:\n {train_data_custom.class_to_idx}")
print(f"Get dataset length:\n {len(train_data_custom)}")

<src.data_helpers.ImageFolderCustom object at 0x0000022A46D29E80> <src.data_helpers.ImageFolderCustom object at 0x0000022A46BBBD90>

Get classes:
 ['pizza', 'steak', 'sushi']
Get class-label dict:
 {'pizza': 0, 'steak': 1, 'sushi': 2}
Get dataset length:
 225


In [7]:
# Turn custom dataset into dataloader
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() - 1

train_dataloader_custom = DataLoader(
    dataset=train_data_custom,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS # how many cores are used to load data
)

test_dataloader_custom = DataLoader(
    dataset=test_data_custom,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print(train_dataloader_custom, test_dataloader_custom)
print(f"Number of train batches: {len(train_dataloader_custom)}")
print(f"Number of test batches: {len(test_dataloader_custom)}")

img, label = next(iter(train_dataloader_custom))
print(f"Image shape: {img.shape} -> [batch size, color channel, height, width]")
print(f"Label shape: {label.shape}")

<torch.utils.data.dataloader.DataLoader object at 0x0000022A46D29FD0> <torch.utils.data.dataloader.DataLoader object at 0x0000022A46DA42D0>
Number of train batches: 8
Number of test batches: 3
Image shape: torch.Size([32, 3, 64, 64]) -> [batch size, color channel, height, width]
Label shape: torch.Size([32])


In [ ]:
# This data can now be used as usual to train a model...